# Inference & Serving — Hands-On

**LLM Engineering · Domain 8 · Roadmap Week 22**

Offline simulations for continuous batching, streaming, KV-cache pressure, quantized memory, and autoscaling signals.

## 0. Setup

In [ ]:
%pip install -q numpy
from collections import deque
import json, numpy as np
rng = np.random.RandomState(46)
print("ok")

## 1. Continuous batching simulator

In [ ]:
def continuous_batch(lengths, max_batch):
    q = deque([{"id": i, "remaining": int(t), "start": None, "finish": None} for i,t in enumerate(lengths)])
    active, timeline, step, done = [], [], 0, []
    while q or active:
        while q and len(active) < max_batch:
            r = q.popleft(); r["start"] = step; active.append(r)
        timeline.append(len(active))
        for r in active: r["remaining"] -= 1
        keep = []
        for r in active:
            if r["remaining"] == 0: r["finish"] = step+1; done.append(r)
            else: keep.append(r)
        active = keep; step += 1
    return timeline, done
timeline, done = continuous_batch([6,2,9,1,4,5,3], 3)
print("active_by_step", timeline)
print("mean_batch", round(float(np.mean(timeline)), 2), "steps", len(timeline))

## 2. Throughput vs max batch

In [ ]:
for b in [1,2,4,8]:
    tl, done = continuous_batch(rng.randint(2, 12, size=30), b)
    total_tokens = sum(tl)
    print(f"batch={b} steps={len(tl):>3} tokens/sec_unit={total_tokens/len(tl):.2f} utilization~{np.mean(tl)/b:.2f}")

## 3. KV-cache memory and GQA

In [ ]:
def kv_gb(layers, d_model, seq, batch, bytes_per=2, kv_heads=None, heads=None):
    frac = 1.0 if kv_heads is None else kv_heads/heads
    return 2*layers*d_model*seq*batch*bytes_per*frac/1e9
for seq in [2048, 8192, 32768]:
    print(seq, "MHA", round(kv_gb(32,4096,seq,8), 1), "GB", "GQA", round(kv_gb(32,4096,seq,8,kv_heads=8,heads=32), 1), "GB")

## 4. Quantized serving memory

In [ ]:
params = 7_000_000_000
for bits in [16,8,4]:
    print(f"{bits}-bit weights ~= {params*bits/8/1e9:.1f} GB before overhead")

## 5. SSE streaming chunks

In [ ]:
def fake_tokens():
    for t in ["Continuous", " batching", " improves", " throughput", "."]: yield t
for token in fake_tokens(): print("data:", json.dumps({"token": token}))
print("data: [DONE]")

## 6. Autoscaling signal from queue and p95

In [ ]:
queue_depth = np.array([2,4,8,13,21,18,9])
ttft_ms = np.array([180,220,290,430,760,690,340])
kv_util = np.array([.42,.50,.61,.74,.88,.84,.66])
scale_out = (queue_depth[-1] > 10) or (np.percentile(ttft_ms, 95) > 650) or (kv_util[-1] > .85)
print("p95_ttft", np.percentile(ttft_ms,95), "scale_out", bool(scale_out))

## 7. Exercises
1. Add a max queue-wait timeout to the scheduler.
2. Separate prefill cost from decode cost in the simulation.
3. Add a cancellation event that stops a stream halfway.
4. Estimate max concurrency under a 40 GB GPU budget.

## Links
- Literature note: `02 Literature Notes/LLM Engineering/Inference and Serving`
- Builds on: `02 Literature Notes/LLM Engineering/LLM Efficiency`